# 🧠 KV Cache Compression with TurboQuant
### Notebook 02 — LLM Perplexity & Memory Benchmark · **Requires GPU**

> **Run this on Google Colab with A100 (recommended) or T4 (use Gemma-2-2B)**

**What this notebook demonstrates:**
1. ✅ Load a real HuggingFace LLM (Gemma-2-2B or Llama-3.1-8B)
2. ✅ Measure baseline FP16 perplexity on WikiText-2
3. ✅ Patch the model with TurboQuant KV cache compression (one line!)
4. ✅ Measure compressed perplexity at 4-bit, 3-bit, 2-bit
5. ✅ Compare GPU memory before/after
6. ✅ Side-by-side generation quality comparison
7. ✅ Speed benchmark (tokens/second)

**Expected results (from paper):**
| Method | Bits | Perplexity | Memory Reduction |
|--------|------|------------|------------------|
| Baseline FP16 | 16 | ~8.5 | 1× |
| TurboQuant | 4 | ~8.7 ±0.3 | ~4× |
| TurboQuant | 3 | ~9.0 ±0.5 | ~5× |
| TurboQuant | 2 | ~12+ | ~8× |

In [ ]:
# @title 🔍 Environment Check
import subprocess, sys, torch

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')
    if props.total_memory < 15e9:
        print('⚠️  < 16 GB VRAM detected. Use Gemma-2-2B (not Llama-3.1-8B)')
    else:
        print('✅ Sufficient VRAM for Llama-3.1-8B')
else:
    print('❌ No GPU found. Switch to GPU runtime: Runtime → Change runtime type → GPU')

In [ ]:
# @title 📦 Install Dependencies (run once)
%%capture
!pip install torch==2.3.1 transformers==4.44.2 accelerate==0.33.0
!pip install datasets einops rich tqdm seaborn matplotlib
# Install TurboQuant package
import os
if os.path.exists('/content/TQ-infer-engine'):
    !pip install -e /content/TQ-infer-engine -q
else:
    !git clone https://github.com/Paramveersingh-S/TQ-infer-engine.git /content/TQ-infer-engine
    !pip install -e /content/TQ-infer-engine -q

In [ ]:
# @title 🔐 HuggingFace Login (required for Llama — skip for Gemma)
# from huggingface_hub import login
# login()  # Enter token at https://huggingface.co/settings/tokens
# Request Llama access: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
print('ℹ️  Gemma-2-2B does not require login. Uncomment above for Llama.')

In [ ]:
# @title 📚 Imports
import sys, os, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/content/TQ-infer-engine')
from tqe.algorithms.turbo_quant import TurboQuantizer
from tqe.kv_cache.compressor import KVCacheCompressor

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# Model selection
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if VRAM_GB >= 16:
    MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
else:
    MODEL_NAME = 'google/gemma-2-2b-it'
print(f'Model: {MODEL_NAME}  (VRAM={VRAM_GB:.1f} GB)')

In [ ]:
# @title 🤗 Section 1 — Load Model (Gemma-2-2B or Llama-3.1-8B)
print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'✅ Model loaded: {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters')
if DEVICE == 'cuda':
    print(f'GPU memory after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# @title 📏 Perplexity Evaluation Utility
def compute_perplexity(model, tokenizer, text, max_tokens=2048, device='cuda', stride=512):
    encodings = tokenizer(text, return_tensors='pt')
    seq_len = min(encodings.input_ids.shape[1], max_tokens)
    input_ids = encodings.input_ids[:, :seq_len].to(device)
    nlls, prev_end = [], 0
    for begin in range(0, seq_len, stride):
        end = min(begin + stride, seq_len)
        target_len = end - prev_end
        chunk = input_ids[:, begin:end]
        with torch.no_grad():
            out = model(chunk, labels=chunk)
        nlls.append(out.loss * target_len)
        prev_end = end
        if end >= seq_len:
            break
    ppl = math.exp(torch.stack(nlls).sum().item() / seq_len)
    return ppl

# Load WikiText-2
print('Loading WikiText-2...')
ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
text = '\n'.join(ds['text'])
print(f'Dataset: {len(text):,} characters')

In [ ]:
# @title 📊 Section 2 — Baseline Perplexity (FP16, no compression)
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    baseline_mem_before = torch.cuda.memory_allocated() / 1e9

print('Computing baseline FP16 perplexity...')
t0 = time.perf_counter()
baseline_ppl = compute_perplexity(model, tokenizer, text, max_tokens=2048, device=DEVICE)
baseline_time = time.perf_counter() - t0

if DEVICE == 'cuda':
    baseline_peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f'Peak GPU memory: {baseline_peak_mem:.2f} GB')
else:
    baseline_peak_mem = 0.0

print(f'\n✅ Baseline FP16 Perplexity: {baseline_ppl:.3f}  (time: {baseline_time:.1f}s)')
results_log = [{'method': 'Baseline FP16', 'bits': 16.0, 'ppl': baseline_ppl,
                'peak_mem_gb': baseline_peak_mem, 'ratio': 1.0}]

In [ ]:
# @title ⚡ Section 3 & 4 — TurboQuant Perplexity at 4-bit, 3-bit, 2-bit
for bits in [4.0, 3.0, 2.0]:
    print(f'\n--- TurboQuant {bits:.0f}-bit ---')
    compressor = KVCacheCompressor(model, bits_per_dim=bits, device=DEVICE)
    compressor.patch_model()
    layers = compressor.stats()['num_layers_patched']
    print(f'  Patched {layers} attention layers')

    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    t0  = time.perf_counter()
    ppl = compute_perplexity(model, tokenizer, text, max_tokens=2048, device=DEVICE)
    elapsed = time.perf_counter() - t0

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
    ratio    = compressor.stats()['compression_ratio']
    ppl_delta = ppl - baseline_ppl

    print(f'  Perplexity : {ppl:.3f}  (delta = +{ppl_delta:.3f})')
    print(f'  Peak mem   : {peak_mem:.2f} GB  (ratio = {ratio:.2f}×)')
    print(f'  Time       : {elapsed:.1f}s')

    results_log.append({'method': f'TurboQuant {bits:.0f}bit', 'bits': bits,
                        'ppl': ppl, 'peak_mem_gb': peak_mem, 'ratio': ratio})
    compressor.unpatch_model()

print('\n✅ All TurboQuant benchmarks complete.')

In [ ]:
# @title 💾 Section 5 — Memory Comparison Before/After
# Show theoretical KV cache memory vs compressed
from tqe.utils.memory_utils import estimate_kv_cache_memory

n_layers = 28  # typical for 7B model
n_kv_heads = 8
head_dim = 128
context_lengths = [1024, 4096, 16384, 32768, 65536, 131072]

baseline_gb, compressed_4bit_gb, compressed_2bit_gb = [], [], []

for ctx in context_lengths:
    info = estimate_kv_cache_memory(n_layers, n_kv_heads, head_dim, ctx, dtype_bytes=2)
    baseline_gb.append(info['total_gb'])
    compressed_4bit_gb.append(info['total_gb'] / 4.0)
    compressed_2bit_gb.append(info['total_gb'] / 8.0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(context_lengths, baseline_gb, 'o-', color='#C73E1D', linewidth=2.5,
        markersize=8, label='Baseline FP16')
ax.plot(context_lengths, compressed_4bit_gb, 's-', color='#2E86AB', linewidth=2.5,
        markersize=8, label='TurboQuant 4-bit (~4×)')
ax.plot(context_lengths, compressed_2bit_gb, '^-', color='#44BBA4', linewidth=2.5,
        markersize=8, label='TurboQuant 2-bit (~8×)')
ax.fill_between(context_lengths, compressed_4bit_gb, baseline_gb,
                alpha=0.1, color='#2E86AB', label='Savings (4-bit)')
ax.axhline(16.0, color='orange', linestyle='--', alpha=0.7, label='16 GB GPU limit')
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('KV Cache Memory (GB)', fontsize=12)
ax.set_title(f'KV Cache Memory vs Context Length\n({n_layers} layers, {n_kv_heads} KV heads, head_dim={head_dim})', fontsize=13)
ax.legend(fontsize=10)
ax.set_xscale('log', base=2)
ax.set_xticks(context_lengths)
ax.set_xticklabels([f'{x//1024}K' if x >= 1024 else str(x) for x in context_lengths])
plt.tight_layout()
plt.savefig('kv_memory_vs_context.png', dpi=150)
plt.show()

In [ ]:
# @title 🔤 Section 6 — Generation Quality Comparison (same prompt, baseline vs 4-bit)
PROMPT = """The key innovation in transformer attention is the query-key-value mechanism. 
Specifically, the attention score between token i and token j is computed as"""

inputs = tokenizer(PROMPT, return_tensors='pt').to(DEVICE)

# Baseline
with torch.no_grad():
    baseline_out = model.generate(
        **inputs, max_new_tokens=80, do_sample=False, temperature=1.0
    )
baseline_text = tokenizer.decode(baseline_out[0], skip_special_tokens=True)

# TurboQuant 4-bit
comp = KVCacheCompressor(model, bits_per_dim=4.0, device=DEVICE)
comp.patch_model()
with torch.no_grad():
    tq_out = model.generate(
        **inputs, max_new_tokens=80, do_sample=False, temperature=1.0
    )
tq_text = tokenizer.decode(tq_out[0], skip_special_tokens=True)
comp.unpatch_model()

print('='*70)
print('PROMPT:', PROMPT)
print('='*70)
print('\n🔵 BASELINE FP16 OUTPUT:')
print(baseline_text[len(PROMPT):])
print('\n⚡ TURBOQUANT 4-BIT OUTPUT:')
print(tq_text[len(PROMPT):])
print('='*70)
print('✅ At 4-bit, outputs should be nearly identical!')

In [ ]:
# @title ⚡ Section 7 — Speed Benchmark (tokens/second)
def measure_throughput(model, tokenizer, prompt, n_tokens=100, n_runs=3, device='cuda'):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    times  = []
    for _ in range(n_runs):
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=n_tokens, do_sample=False)
        if device == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    best_time = min(times)
    return n_tokens / best_time  # tokens/sec

PROMPT  = 'TurboQuant is a vector quantization algorithm that'
N_TOKENS = 50

speed_results = {}

# Baseline
tps_base = measure_throughput(model, tokenizer, PROMPT, N_TOKENS, device=DEVICE)
speed_results['FP16'] = tps_base
print(f'Baseline FP16: {tps_base:.1f} tokens/sec')

# TurboQuant at various bits
for bits in [4.0, 3.0, 2.0]:
    c = KVCacheCompressor(model, bits_per_dim=bits, device=DEVICE)
    c.patch_model()
    tps = measure_throughput(model, tokenizer, PROMPT, N_TOKENS, device=DEVICE)
    speed_results[f'TQ {bits:.0f}bit'] = tps
    c.unpatch_model()
    print(f'TurboQuant {bits:.0f}bit: {tps:.1f} tokens/sec ({tps/tps_base:.2f}× speedup)')

fig, ax = plt.subplots(figsize=(8, 5))
labels = list(speed_results.keys())
values = list(speed_results.values())
colors = ['#C73E1D', '#2E86AB', '#A23B72', '#44BBA4']
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.0f}', ha='center', fontweight='bold')
ax.set_ylabel('Tokens per Second')
ax.set_title('Generation Throughput: Baseline vs TurboQuant')
plt.tight_layout()
plt.savefig('kv_speed_benchmark.png', dpi=150)
plt.show()

In [ ]:
# @title 📋 Section 8 — Results Summary Table & Chart
print('\n' + '='*70)
print('  KV CACHE COMPRESSION BENCHMARK RESULTS SUMMARY')
print(f'  Model: {MODEL_NAME}')
print(f'  Dataset: WikiText-2 (2048 tokens)')
print('='*70)
print(f'{'Method':22} {'Bits':>5} {'Perplexity':>12} {'Peak Mem (GB)':>14} {'Ratio':>8}')
print('-'*65)
for r in results_log:
    delta = f"(+{r['ppl']-results_log[0]['ppl']:.2f})" if r['bits'] < 16 else ''
    print(f"{r['method']:22} {r['bits']:>5.1f} {r['ppl']:>12.3f} {delta:>10} "
          f"{r['peak_mem_gb']:>8.2f}GB {r['ratio']:>7.1f}×")
print('='*70)

# Summary chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

methods = [r['method'] for r in results_log]
ppls    = [r['ppl']    for r in results_log]
ratios  = [r['ratio']  for r in results_log]
colors  = ['#C73E1D'] + ['#2E86AB', '#A23B72', '#44BBA4']

ax1.barh(methods, ppls, color=colors[:len(methods)], edgecolor='white')
ax1.axvline(ppls[0], color='red', linestyle='--', alpha=0.6, label='Baseline')
ax1.set_xlabel('Perplexity (lower is better)')
ax1.set_title('WikiText-2 Perplexity')
ax1.legend()

ax2.barh(methods, ratios, color=colors[:len(methods)], edgecolor='white')
ax2.set_xlabel('Compression Ratio (×FP16)')
ax2.set_title('Memory Compression Ratio')

plt.suptitle(f'TurboQuant KV Cache Benchmark — {MODEL_NAME}', fontsize=13)
plt.tight_layout()
plt.savefig('kv_summary.png', dpi=150)
plt.show()

# Save CSV
import csv, os
os.makedirs('results', exist_ok=True)
with open('results/kv_cache_benchmark.csv', 'w') as f:
    writer = csv.DictWriter(f, fieldnames=results_log[0].keys())
    writer.writeheader()
    writer.writerows(results_log)
print('\n📄 Saved: results/kv_cache_benchmark.csv')